# Python Context Managers and Cleanup

As we venture deeper into the fascinating world of Python, we stumble upon a powerful feature known as context managers. Context managers allow us to properly manage resources so we can avoid leaving open connections or having locked resources. They're ideal for handling setup and cleanup procedures, which might be difficult to manage due to exceptions or multiple return paths.

Let's delve in!



## Understanding Context Managers and the `with` Statement

In Python, context managers are typically used with the `with` keyword. You may have already seen this when reading or writing to files:


In [ ]:
with open('example.txt', 'r') as my_file:
    content = my_file.read()



In this code, `open('example.txt', 'r')` is a context manager that handles the opening and closing of the file. We don't need to remember to close the file — it's taken care of us automatically!


<div style="background: var(--vscode-textBlockQuote-background, var(--jp-layout-color2, rgba(74,144,217,0.08))); border-left: 4px solid var(--vscode-textLink-foreground, var(--jp-brand-color1, #4a90d9)); color: var(--vscode-foreground, inherit); padding: 12px 16px; border-radius: 0 4px 4px 0; margin: 12px 0; max-width: 100%; box-sizing: border-box;">
<strong>Try It Out: Read and Write Files with Context Managers</strong>
<pre style="background:transparent; white-space:pre-wrap; overflow-x:auto; margin:8px 0 0 0;">Use with statements to:
  1. Write two lines to a file called 'notes.txt'
  2. Read the file back and print its contents
The file should close automatically after each block — no need to call f.close().</pre>
</div>

In [ ]:
# Write to a file using a context manager
with open('notes.txt', ___) as f:
    f.write(___)

# Read it back using another context manager
with open('notes.txt', ___) as f:
    content = f.___()  # read entire contents
    print(content)

<details>
<summary><strong>Show Answer</strong></summary>

```python
with open('notes.txt', 'w') as f:
    f.write('Context managers handle cleanup automatically.\n')
    f.write('No need to call close().')

with open('notes.txt', 'r') as f:
    content = f.read()
    print(content)
```

</details>


## How Context Managers Work

The magic of context managers is in two special methods: `__enter__` and `__exit__`.

The `__enter__` method is executed at the beginning of the `with` block. In the file example above, this method opens the file and returns it.

The `__exit__` method is executed at the end of the `with` block — even if the block is exited due to an exception. This method takes care of the cleanup. In the file example, it closes the file.



## Creating Your Own Context Managers

We aren't limited to using built-in context managers. Python gives us the power to create our own!



### Using Classes

We can create a context manager by defining a class with `__enter__` and `__exit__` methods. For example, let's create a simple context manager that logs the entering and exiting of a code block:


In [ ]:
class LogContext:
    def __enter__(self):
        print("Entering the block")

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Exiting the block")

with LogContext():
    print("Hello, World!")


### `__exit__` arguments 

Let's break down what each of these arguments to the `__exit__` method mean:

- `exc_type`: The type of the exception that was raised. If no exception was raised, it will be `None`.

- `exc_val`: An instance of the exception that was raised. Its value is typically an error message. If no exception was raised, it will be `None`.

- `exc_tb`: A traceback object encapsulating the call stack at the point where the exception was raised. If no exception was raised, it will be `None`.

These arguments give you information about any exception that might have happened inside the `with` block. They allow your `__exit__` method to respond differently to different kinds of exceptions, if you want it to.

For example, you might want to log different messages for different types of exceptions, or you might want to re-raise the exception after logging it. Here's an example:


In [ ]:
class LogContext:
    def __enter__(self):
        print("Entering the block")

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Exiting the block")

        if exc_type is not None:
            print(f"Exception has been handled, {exc_val}")
            
        return True  # Exception has been handled



In this version of the `ManagedFile` class, the `__exit__` method logs a message if an exception was raised, and then returns `True` to indicate that it has handled the exception. This will prevent the exception from being propagated further.

<div style="background: var(--vscode-textBlockQuote-background, var(--jp-layout-color2, rgba(74,144,217,0.08))); border-left: 4px solid var(--vscode-textLink-foreground, var(--jp-brand-color1, #4a90d9)); color: var(--vscode-foreground, inherit); padding: 12px 16px; border-radius: 0 4px 4px 0; margin: 12px 0; max-width: 100%; box-sizing: border-box;">
<strong>Try It Out: Build a Class-Based Context Manager</strong>
<pre style="background:transparent; white-space:pre-wrap; overflow-x:auto; margin:8px 0 0 0;">Create a class Timer that acts as a context manager:
  - __enter__ records the start time using time.time() and returns self
  - __exit__ computes elapsed time and prints it as 'Block took X.XXXX seconds'
  - __exit__ should return False (do not suppress exceptions)

Wrap a loop that computes sum(i**2 for i in range(1_000_000)) inside with Timer().</pre>
</div>

In [ ]:
import time

class Timer:
    def __enter__(self):
        self.start = ___
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = ___ - self.start
        print("Block took " + str(round(elapsed, 4)) + " seconds")
        return False

with Timer():
    total = sum(i ** 2 for i in range(1_000_000))
    print("Sum of squares:", total)

<details>
<summary><strong>Show Answer</strong></summary>

```python
import time

class Timer:
    def __enter__(self):
        self.start = time.time()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = time.time() - self.start
        print("Block took " + str(round(elapsed, 4)) + " seconds")
        return False

with Timer():
    total = sum(i ** 2 for i in range(1_000_000))
    print("Sum of squares:", total)
```

</details>

### Using Generators

Python also allows us to create a context manager using generators and the `@contextlib.contextmanager` decorator from the `contextlib` module. This can be a simpler way to create a context manager:


In [ ]:
from contextlib import contextmanager

@contextmanager
def log_context():
    print("Entering the block")
    yield
    print("Exiting the block")

with log_context():
    print("Hello, World!")


This will output the same as before.


<div style="background: var(--vscode-textBlockQuote-background, var(--jp-layout-color2, rgba(74,144,217,0.08))); border-left: 4px solid var(--vscode-textLink-foreground, var(--jp-brand-color1, #4a90d9)); color: var(--vscode-foreground, inherit); padding: 12px 16px; border-radius: 0 4px 4px 0; margin: 12px 0; max-width: 100%; box-sizing: border-box;">
<strong>Try It Out: Generator-Based Context Manager with @contextmanager</strong>
<pre style="background:transparent; white-space:pre-wrap; overflow-x:auto; margin:8px 0 0 0;">Use @contextlib.contextmanager to create a suppress_errors() context manager
that catches any exception raised inside the with block, prints
'Suppressed error: <message>', and continues without re-raising.

Test it: inside the with block print 'Before error', then raise a ValueError.</pre>
</div>

In [ ]:
from contextlib import contextmanager

@contextmanager
def suppress_errors():
    try:
        ___  # yield control to the with block
    except Exception as e:
        print("Suppressed error:", e)

with suppress_errors():
    print("Before error")
    raise ValueError("Something went wrong!")
    print("This line will NOT print")

print("Execution continues after the with block")

<details>
<summary><strong>Show Answer</strong></summary>

```python
from contextlib import contextmanager

@contextmanager
def suppress_errors():
    try:
        yield
    except Exception as e:
        print("Suppressed error:", e)

with suppress_errors():
    print("Before error")
    raise ValueError("Something went wrong!")
    print("This line will NOT print")

print("Execution continues after the with block")
```

</details>

## Using arguments in a context manager

Let's make a context manager using a class that accepts arguments. 

Let's design a context manager called `ManagedFile` that will open a file, perform operations, and then ensure that the file is closed. This context manager will accept the filename and the mode of opening as arguments.

Here's how we might implement it:


In [ ]:
class ManagedFile:
    def __init__(self, filename, mode):
        self.filename = filename
        self.mode = mode

    def __enter__(self):
        self.file = open(self.filename, self.mode)
        return self.file

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            self.file.close()

# Usage:
with ManagedFile('hello.txt', 'w') as f:
    f.write('Hello, world!')
    f.write('Python context manager is great!')


In the `__init__` method, we store the filename and mode. Then, in the `__enter__` method, we open the file with the given mode and return it. In the `__exit__` method, we close the file.

Within the `with` block, we can do whatever we want with `f`. When the `with` block is exited (either normally or due to an exception), the `__exit__` method is called, and the file is closed.

<div style="background: var(--vscode-textBlockQuote-background, var(--jp-layout-color2, rgba(74,144,217,0.08))); border-left: 4px solid var(--vscode-textLink-foreground, var(--jp-brand-color1, #4a90d9)); color: var(--vscode-foreground, inherit); padding: 12px 16px; border-radius: 0 4px 4px 0; margin: 12px 0; max-width: 100%; box-sizing: border-box;">
<strong>Try It Out: Context Manager with Constructor Arguments</strong>
<pre style="background:transparent; white-space:pre-wrap; overflow-x:auto; margin:8px 0 0 0;">Create a class SectionLogger(section_name) that acts as a context manager:
  - Stores section_name in __init__
  - __enter__ prints 'Starting: <section_name>' and returns self
  - __exit__ prints 'Finished: <section_name>' and returns False

Use it to wrap two separate code blocks with different section names.</pre>
</div>

In [ ]:
class SectionLogger:
    def __init__(self, section_name):
        self.___ = section_name

    def __enter__(self):
        print("Starting: " + ___)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Finished: " + ___)
        return False

with SectionLogger("Data Loading"):
    data = [1, 2, 3, 4, 5]
    print("Loaded", len(data), "items")

with SectionLogger("Processing"):
    result = sum(data)
    print("Sum:", result)

<details>
<summary><strong>Show Answer</strong></summary>

```python
class SectionLogger:
    def __init__(self, section_name):
        self.section_name = section_name

    def __enter__(self):
        print("Starting: " + self.section_name)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Finished: " + self.section_name)
        return False

with SectionLogger("Data Loading"):
    data = [1, 2, 3, 4, 5]
    print("Loaded", len(data), "items")

with SectionLogger("Processing"):
    result = sum(data)
    print("Sum:", result)
```

</details>


## Practice!

Context managers are everywhere in Python, not just with file operations. They're used with thread locking, changing the current directory, and more. Try to use context managers whenever you need to manage resources or change some global state temporarily. You can also practice creating your own context managers with classes or generators!

That wraps up our overview of context managers in Python. They're a powerful tool that can help us write cleaner and more efficient code by handling setup and cleanup automatically. Happy coding!

<div style="background: var(--vscode-textBlockQuote-background, var(--jp-layout-color2, rgba(74,144,217,0.08))); border-left: 4px solid var(--vscode-textLink-foreground, var(--jp-brand-color1, #4a90d9)); color: var(--vscode-foreground, inherit); padding: 12px 16px; border-radius: 0 4px 4px 0; margin: 12px 0; max-width: 100%; box-sizing: border-box;">
<strong>Practice Problem 1: File Transaction Logger</strong>
<pre style="background:transparent; white-space:pre-wrap; overflow-x:auto; margin:8px 0 0 0;">Build a class-based context manager FileTransaction(filename) that:

  __enter__: opens the file in append mode ('a') and returns the file object

  __exit__:
    - If no exception (exc_type is None): writes 'COMMITTED\n' then closes
    - If an exception occurred: writes 'ROLLED BACK\n' then closes, and
      returns True to suppress the exception

Test 1 (success): open the context, write a log line — should commit
Test 2 (failure): open the context, write a line, then raise RuntimeError —
                  should roll back and NOT propagate the error
After both tests, read and print the full contents of the log file.</pre>
</div>

In [ ]:
class FileTransaction:
    def __init__(self, filename):
        self.filename = filename
        self.file = None

    def __enter__(self):
        self.file = open(self.filename, ___)
        return self.file

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is None:
            self.file.write("COMMITTED\n")
        else:
            self.file.write("ROLLED BACK\n")
        self.file.close()
        return ___  # suppress exceptions

# Test 1: successful transaction
with FileTransaction("transaction.log") as f:
    f.write("Log entry: data saved successfully\n")

# Test 2: failed transaction
with FileTransaction("transaction.log") as f:
    f.write("Log entry: starting risky operation\n")
    raise RuntimeError("disk full")

# Read and print the log
with open("transaction.log") as f:
    print(f.read())

<details>
<summary><strong>Show Answer</strong></summary>

```python
class FileTransaction:
    def __init__(self, filename):
        self.filename = filename
        self.file = None

    def __enter__(self):
        self.file = open(self.filename, 'a')
        return self.file

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is None:
            self.file.write("COMMITTED\n")
        else:
            self.file.write("ROLLED BACK\n")
        self.file.close()
        return True

with FileTransaction("transaction.log") as f:
    f.write("Log entry: data saved successfully\n")

with FileTransaction("transaction.log") as f:
    f.write("Log entry: starting risky operation\n")
    raise RuntimeError("disk full")

with open("transaction.log") as f:
    print(f.read())
```

</details>

<div style="background: var(--vscode-textBlockQuote-background, var(--jp-layout-color2, rgba(74,144,217,0.08))); border-left: 4px solid var(--vscode-textLink-foreground, var(--jp-brand-color1, #4a90d9)); color: var(--vscode-foreground, inherit); padding: 12px 16px; border-radius: 0 4px 4px 0; margin: 12px 0; max-width: 100%; box-sizing: border-box;">
<strong>Practice Problem 2: Generator Context Manager — Connection Simulator</strong>
<pre style="background:transparent; white-space:pre-wrap; overflow-x:auto; margin:8px 0 0 0;">Use @contextmanager to create managed_connection(host) that:

  - Prints 'Connecting to <host>...'
  - Yields a mock connection string: 'conn://' + host
  - In a finally block (so it always runs), prints 'Disconnecting from <host>'

Test 1 (normal): use the connection to print 'Sending data via <conn>'
Test 2 (error): use the connection, then raise a ConnectionError mid-block.
                Wrap the second with block in try/except to handle the error.
Verify that 'Disconnecting' is printed even when an exception occurs.</pre>
</div>

In [ ]:
from contextlib import contextmanager

@contextmanager
def managed_connection(host):
    print("Connecting to " + host + "...")
    conn = ___
    try:
        ___  # yield the connection string
    finally:
        print("Disconnecting from " + host)

# Test 1: normal use
with managed_connection("db.example.com") as conn:
    print("Sending data via", conn)

print()

# Test 2: exception inside the block
try:
    with managed_connection("api.example.com") as conn:
        print("Connected via", conn)
        raise ConnectionError("Network timeout")
except ConnectionError as e:
    print("Caught:", e)

<details>
<summary><strong>Show Answer</strong></summary>

```python
from contextlib import contextmanager

@contextmanager
def managed_connection(host):
    print("Connecting to " + host + "...")
    conn = 'conn://' + host
    try:
        yield conn
    finally:
        print("Disconnecting from " + host)

with managed_connection("db.example.com") as conn:
    print("Sending data via", conn)

print()

try:
    with managed_connection("api.example.com") as conn:
        print("Connected via", conn)
        raise ConnectionError("Network timeout")
except ConnectionError as e:
    print("Caught:", e)
```

</details>